# 03 — Development execution and recovery

Thin representative-matrix interface to run_development.py. Default execution reads existing evidence and displays a command; matrix training requires explicit opt-in.

## Environment and artifact setup

Use Python 3.12 as documented in the repository. Install the pinned PyTorch requirement (requirements-model.txt from the CPU index, or requirements-cuda.txt from the cu126 index), plus requirements-dev.txt. Notebook presentation dependencies used for recorded validation: nbformat 5.10.4, nbclient 0.10.2, ipykernel 6.29.5, pandas 3.0.1, matplotlib 3.10.8.

Open from the repository or set HUNG_REPO_ROOT. Download/extract identified inputs and run artifacts; set HUNG_INPUT_BASE and HUNG_RUN_ROOT if locations differ. Missing files fail explicitly. Large inputs/checkpoints stay outside Git; only load trusted project checkpoints.

Default cells inspect existing evidence and run small CPU examples. They do not launch matrix training. Recorded matrix code SHA: a34a49e5f04920ef30456fe6c0ea56eb2f1d659a, diagnostic config v1 and selection v2. Clean CPU reproduction was validated in a new isolated repo-local Python 3.13.7 environment with torch 2.8.0+cpu, pip check, the full unit suite and a fresh kernel per notebook. README's reference Python 3.12 environment was not executed in this local check. The global CUDA environment has an unrelated ultralytics/numpy conflict; it was not inherited by the isolated environment.


In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path

import pandas as pd

candidates = (
    [Path(os.environ["HUNG_REPO_ROOT"])]
    if os.environ.get("HUNG_REPO_ROOT")
    else [Path.cwd(), *Path.cwd().parents]
)
REPO = next(
    (
        path.resolve()
        for path in candidates
        if (path / "experiments/hung_sprint2/model.py").is_file()
    ),
    None,
)
if REPO is None:
    raise FileNotFoundError("Set HUNG_REPO_ROOT to the checked-out repository.")
sys.path.insert(0, str(REPO))
RUN_ROOT = Path(
    os.environ.get("HUNG_RUN_ROOT", str(REPO / "runs/hung-39-representative-local-20261025T2103"))
).resolve()
INPUT_BASE = Path(
    os.environ.get("HUNG_INPUT_BASE", str(REPO / "data/inputs/issue18-input-handoff-v1-20261019"))
).resolve()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


identity = read_json(RUN_ROOT / "matrix_identity.json")
config = read_json(REPO / "experiments/hung_sprint2/development_config_v1.json")
assert identity["phase"] == "representative"
assert identity["config_sha256"] == sha256(
    REPO / "experiments/hung_sprint2/development_config_v1.json"
)
assert identity["selection_sha256"] == sha256(
    REPO / "docs/reports/hung-development-selection-20261025-v2.json"
)
print(
    {
        "run_id": RUN_ROOT.name,
        "execution_code_sha": identity["code"]["commit"],
        "device": identity["device"],
    }
)

{'run_id': 'hung-39-representative-local-20261025T2103', 'execution_code_sha': 'a34a49e5f04920ef30456fe6c0ea56eb2f1d659a', 'device': 'cuda'}


## Platform setup

Reuse [Kaggle bundle/input setup](../kaggle_39_verification.ipynb) and [Colab/Drive setup](../colab_39_verification.ipynb), preserving their historical pinned verification identities. This notebook consumes a verified checkout, inputs and artifacts rather than copying training code.

Choose HUNG_BACKEND=local|kaggle|colab and HUNG_DEVICE=cpu|cuda. Before a new execution, verify quota, CUDA tensor allocation, ZIP/manifest hashes and free disk. Stop on CUDA failure; every backend execution requires a new output root and identity. Keep the 6/8/9 entity selection and seeds 7/11/19 fixed.

In [2]:
import platform
import shutil
import subprocess

import torch

backend = os.environ.get("HUNG_BACKEND", "local")
device = os.environ.get("HUNG_DEVICE", "cuda")
if backend not in {"local", "kaggle", "colab"} or device not in {"cpu", "cuda"}:
    raise ValueError("Unsupported backend/device")
selection_path = REPO / "docs/reports/hung-development-selection-20261025-v2.json"
config_path = REPO / "experiments/hung_sprint2/development_config_v1.json"
selection = read_json(selection_path)
pd.DataFrame(
    [
        {
            "budget": budget,
            "entities": len(selection["budgets"][str(budget)]["representative_entities"]),
            "seeds": 3,
            "expected_stages": 3
            * (1 + 3 * len(selection["budgets"][str(budget)]["representative_entities"])),
        }
        for budget in (7, 14, 28)
    ]
)

,budget,entities,seeds,expected_stages
0,7,6,3,57
1,14,8,3,75
2,28,9,3,84


In [3]:
print(
    {
        "backend_requested": backend,
        "device_requested": device,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "free_disk_bytes": shutil.disk_usage(REPO).free,
        "training_launched": False,
    }
)
new_output = os.environ.get("HUNG_NEW_OUTPUT_ROOT")
command = [
    sys.executable,
    "-m",
    "experiments.hung_sprint2.run_development",
    "--input-base",
    str(INPUT_BASE),
    "--selection",
    str(selection_path),
    "--config",
    str(config_path),
    "--output-root",
    new_output or "<new-create-only-output-root>",
    "--phase",
    "representative",
    "--device",
    device,
    "--budgets",
    "7",
    "14",
    "28",
    "--seeds",
    "7",
    "11",
    "19",
    "--execute",
]
print(subprocess.list2cmdline(command))
completed = list(RUN_ROOT.rglob("run_manifest.json"))
pd.DataFrame(
    [{"stage": path.parent.name, "status": read_json(path)["status"]} for path in completed]
).groupby(["stage", "status"]).size().rename("count").to_frame()

{'backend_requested': 'local', 'device_requested': 'cuda', 'python': '3.13.7', 'torch': '2.8.0+cpu', 'cuda_available': False, 'free_disk_bytes': 59564105728, 'training_launched': False}
"D:\Đồ Án\.cache\sprint2-completion\.venv-handoff\Scripts\python.exe" -m experiments.hung_sprint2.run_development --input-base "D:\Đồ Án\.cache\sprint2-completion\data\inputs\issue18-input-handoff-v1-20261019" --selection "D:\Đồ Án\.cache\sprint2-completion\docs\reports\hung-development-selection-20261025-v2.json" --config "D:\Đồ Án\.cache\sprint2-completion\experiments\hung_sprint2\development_config_v1.json" --output-root <new-create-only-output-root> --phase representative --device cuda --budgets 7 14 28 --seeds 7 11 19 --execute


,,count
stage,status,
replay-42,completed,69
replay-90,completed,69
source,completed,9
target,completed,69


## Explicit new execution and recovery

Set HUNG_EXECUTE_NEW_RUN=YES and HUNG_NEW_OUTPUT_ROOT to a **new** directory after preflight/quota checks. The cell refuses an existing root. Recover the recorded execution with its exact original CLI command and pinned code/config/input/device identity from matrix_identity.json: the driver reuses completed matching stages and refuses partial stage directories. Preserve partial outputs before choosing a new root; never combine different backend identities.

Source is shared within one budget/seed. Every entity proceeds through target → replay 42 → replay 90; incomplete-label days remain unscored.

In [4]:
if os.environ.get("HUNG_EXECUTE_NEW_RUN") == "YES":
    if not new_output:
        raise ValueError("Set HUNG_NEW_OUTPUT_ROOT explicitly.")
    output_path = Path(new_output).resolve()
    if output_path.exists():
        raise FileExistsError("New execution requires an unused output root.")
    if device == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA unavailable; execution stopped.")
        probe = torch.ones(1, device="cuda")
        torch.cuda.synchronize()
        print({"actual_probe_device": str(probe.device), "gpu": torch.cuda.get_device_name(0)})
    subprocess.run(command, cwd=REPO, check=True)
else:
    print("Execution opt-in is off. Existing artifacts were read; training was not launched.")

Execution opt-in is off. Existing artifacts were read; training was not launched.


## Handoff

Use notebook 04 after completion to verify 216 stages, checkpoint/journal hashes and 69 replay rows. Archive without replacing old archives; verify CRC and SHA after readback. Retain code/config/selection/input identities. Technical handoff leaves review and Final gates open.